# init

> Router initialization for the structure decomposition workflow

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
from typing import List

from fasthtml.common import APIRouter

# Import subpackage router assemblies
from cjm_fasthtml_workflow_transcript_decomp.routes.core.init import init_core_routers
from cjm_transcript_source_select.routes.init import init_selection_routers
from cjm_transcript_segmentation.routes.init import init_segmentation_routers
from cjm_transcript_vad_align.routes.init import init_alignment_routers

# Import wrapped handlers for cross-domain coordination
from cjm_fasthtml_workflow_transcript_decomp.combined.handlers import (
    wrapped_seg_split, wrapped_seg_merge,
    wrapped_seg_undo, wrapped_seg_reset, wrapped_seg_ai_split,
    create_seg_init_chrome_wrapper, create_align_init_chrome_wrapper,
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

## Router Assembly

The `init_routers` function calls all focused router initializers and wires up cross-router dependencies.

In [ ]:
#| export
def init_routers(
    workflow: "StructureDecompWorkflow",  # The workflow instance
) -> List[APIRouter]:  # List of configured routers
    """Initialize and return all workflow routers."""
    base_prefix = workflow.config.route_prefix

    # Initialize focused routers
    core_routers, core_routes = init_core_routers(
        workflow, f"{base_prefix}/core"
    )
    
    # Selection routers use dependency injection
    selection_routers, selection_urls, selection_routes = init_selection_routers(
        state_store=workflow.state_store,
        source_service=workflow.source_service,
        workflow_id=workflow.config.workflow_id,
        prefix=f"{base_prefix}/selection",
    )

    # Alignment routers (need to initialize first to get align_urls for seg init)
    # Create the align init wrapper first
    wrapped_align_init = create_align_init_chrome_wrapper()
    
    align_routers, align_urls, align_routes = init_alignment_routers(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        source_service=workflow.source_service,
        alignment_service=workflow.alignment_service,
        prefix=f"{base_prefix}/align",
        audio_src_url=core_routes["audio_src"].to(),
        wrapped_init=wrapped_align_init,
    )
    
    # Create the seg init wrapper with URLs needed for KB system
    wrapped_seg_init = create_seg_init_chrome_wrapper(
        align_urls=align_urls,
        switch_chrome_url=core_routes["switch_chrome"].to(),
    )
    
    # Wrapped handlers for cross-domain coordination (alignment status OOB)
    seg_wrapped = {
        "init": wrapped_seg_init,
        "split": wrapped_seg_split,
        "merge": wrapped_seg_merge,
        "undo": wrapped_seg_undo,
        "reset": wrapped_seg_reset,
        "ai_split": wrapped_seg_ai_split,
    }
    
    # Segmentation routers use dependency injection
    seg_routers, seg_urls, seg_routes = init_segmentation_routers(
        state_store=workflow.state_store,
        workflow_id=workflow.config.workflow_id,
        source_service=workflow.source_service,
        segmentation_service=workflow.segmentation_service,
        prefix=f"{base_prefix}/seg",
        max_history_depth=workflow.config.max_history_depth,
        wrapped_handlers=seg_wrapped,
    )

    # Store URL bundles on workflow for renderer access
    workflow._selection_urls = selection_urls
    workflow._seg_urls = seg_urls
    workflow._align_urls = align_urls
    workflow._switch_chrome_url = core_routes["switch_chrome"].to()

    # Store route dicts on workflow
    workflow._core_routes = core_routes
    workflow._selection_routes = selection_routes
    workflow._segmentation_routes = seg_routes
    workflow._alignment_routes = align_routes

    return core_routers + selection_routers + seg_routers + align_routers

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()